# Testing the `Output Guardrail` n8n Workflow

Test cases are loaded from `unit_tests/output_guardrail/*/test.json`.
Each file contains a pre-built `input` dict and an `expectedOutput` with
boolean guardrail flags.

### Request payload shape
| Field | Meaning |
|---|---|
| `replyMessage` | The bot reply to validate |
| `prevAIMessage` | The previous bot message (plain text) |
| `UserMessage` | The user message that prompted the reply |

### Response payload shape
| Field | Type | Meaning |
|---|---|---|
| `fail_outputGuardrail` | `bool` | `true` if **any** check fires |
| `preCheckViolations` | `string` | Comma-separated code-level violations, or empty |
| `personalData` | `bool` | Reply reveals customer PII |
| `nsfw` | `bool` | Reply contains offensive content |
| `hallucinationHarm` | `bool` | Reply makes false financial guarantees |

### URL note
- Test URL (n8n editor open): `{base}/webhook-test/{path}`
- Production URL (workflow Active): `{base}/webhook/{path}`

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests

## 1. Configuration

In [ ]:
N8N_BASE_URL = "https://alphamakeathon-automation.arisetech.dev"  # <-- change me
WEBHOOK_PATH = "efab08a3-b42d-45e4-b424-99ea86faa364"
USE_TEST_URL = False  # True -> use the "Listen for test event" URL instead

CHECK_COLS = ["fail_outputGuardrail", "personalData", "nsfw", "hallucinationHarm"]


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


get_webhook_url()

## 2. Load test cases from JSON

In [ ]:
def _find_project_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "unit_tests").is_dir():
            return p
        p = p.parent
    raise RuntimeError("Could not find project root (directory containing 'unit_tests/')")


def load_unit_tests(agent_type: str) -> list[dict]:
    root = _find_project_root()
    test_dir = root / "unit_tests" / agent_type
    tests = []
    for tc_dir in sorted(test_dir.iterdir()):
        test_file = tc_dir / "test.json"
        if test_file.is_file():
            with open(test_file, encoding="utf-8") as f:
                tests.append(json.load(f))
    print(f"Loaded {len(tests)} test cases from {test_dir}")
    return tests


TEST_CASES = load_unit_tests("output_guardrail")
for tc in TEST_CASES:
    print(f"  {tc['testId']}: {tc['testDescription']}")

## 3. Webhook caller

In [ ]:
def call_output_guardrail(payload: dict, timeout: int = 30) -> dict:
    """POST the pre-built input dict to the Output Guardrail webhook."""
    url = get_webhook_url()
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

## 4. Run a single example

Inspect one test case and confirm connectivity before running the full suite.

In [ ]:
sample_tc = TEST_CASES[0]
print(f"Test: {sample_tc['testId']} — {sample_tc['testDescription']}")
print("\nInput payload:")
print(json.dumps(sample_tc["input"], ensure_ascii=False, indent=2))
print("\nExpected output:")
print(json.dumps(sample_tc["expectedOutput"], ensure_ascii=False, indent=2))

# Uncomment once N8N_BASE_URL points to a real, reachable instance:
# result = call_output_guardrail(sample_tc["input"])
# print("\nActual output:")
# print(json.dumps(result, ensure_ascii=False, indent=2))

## 5. Run all test cases and compare guardrail flags

In [ ]:
rows = []
for tc in TEST_CASES:
    expected = tc["expectedOutput"]
    try:
        result = call_output_guardrail(tc["input"])
        mismatches = [k for k in CHECK_COLS if bool(expected.get(k)) != bool(result.get(k))]
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "pass": len(mismatches) == 0,
            "mismatched_fields": ", ".join(mismatches) if mismatches else "",
            **{f"exp_{k}": expected.get(k) for k in CHECK_COLS},
            **{f"act_{k}": result.get(k) for k in CHECK_COLS},
            "exp_preCheckViolations": expected.get("preCheckViolations", ""),
            "act_preCheckViolations": result.get("preCheckViolations", ""),
            "error": None,
        })
    except requests.exceptions.RequestException as exc:
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "pass": False,
            "mismatched_fields": "",
            **{f"exp_{k}": expected.get(k) for k in CHECK_COLS},
            **{f"act_{k}": None for k in CHECK_COLS},
            "exp_preCheckViolations": expected.get("preCheckViolations", ""),
            "act_preCheckViolations": None,
            "error": str(exc),
        })

results_df = pd.DataFrame(rows)
summary_cols = ["testId", "scenario", "messageMode", "pass", "mismatched_fields",
                "exp_preCheckViolations", "act_preCheckViolations", "error"]
print(f"Passed: {results_df['pass'].sum()}/{len(results_df)}")
results_df[summary_cols]

## Notes

- This workflow is **Active**, so the production URL is
  `{base_url}/webhook/efab08a3-b42d-45e4-b424-99ea86faa364`.
- `preCheckViolations` is set by a code node (deterministic, not LLM), so it
  is always `"empty_output"` for blank replies and `"length_exceeded"` for
  over-length ones. When it is non-empty the LLM guardrail does not run,
  so `personalData`/`nsfw`/`hallucinationHarm` all return `false`.
- `fail_outputGuardrail` is `true` whenever **any** check fires.
- Add more test cases by running the unit-test generator; they are picked up
  automatically the next time this notebook runs.